In [97]:
from dotenv import load_dotenv

load_dotenv()

True

In [98]:

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph, END 

from pydantic import BaseModel, Field

from typing_extensions import TypedDict, Literal

In [99]:
model = "gpt-4o-mini"
agent = ChatOpenAI(model=model, temperature=0.0, max_tokens=500)

In [100]:

class AgentState(TypedDict):
    # User and context
    customer_id: str
    query: str
    conversation_history: list[BaseMessage]
    
    # Intent classification
    intent: Literal["account", "transaction", "general", "unknown"]
    confidence: float
    
    # Agent processing
    response: str
    conversation_history: list[dict[str, str]]
    escalation_requested: bool
    
    # Additional context
    agent_response: str
    

In [101]:
def load_memory_node(state: AgentState) -> AgentState:
    """
    Load customer context and conversation history.
    In a real system, this would query a database or memory store.
    """
    # TODO: Replace with actual memory retrieval
    # For now, just initialize empty if not present
    if not state.get("conversation_history"):
        state["conversation_history"] = []

    print(f"[Memory] Loaded context for customer {state['customer_id']}")
    return state

In [102]:
class IntentClassifier(BaseModel):
    user_intent: Literal["account", "transaction", "general", "unknown"] = Field(
        ...,
        description='Classify whether the user wants to request information about their credit limit (account), check their latest transactions (transation), ask a general question (general), or if the intent is unknown (unknown).'
    )

In [103]:
def classify_intent(state: AgentState) -> AgentState:
    
    prompt = """
    You are an intent classification model. 
    Classify whether the user wants to request information about their credit limit (account), 
    check their latest transactions (transaction), 
    ask a general question (general), or if the intent is unknown (unknown)."""
    
    llm = ChatOpenAI(model=model, temperature=0.0, max_tokens=500)
    
    intent_cls_llm = llm.with_structured_output(IntentClassifier)
    
    intent_result = intent_cls_llm.invoke([
        {
            "role": "system",
            "content": prompt
        },
        {
            "role": "user",
            "content": state["query"]
        }
    ]
    )
    
    state["intent"] = intent_result.user_intent
    state["confidence"] = 1.0
    
    print(f"[Intent classifier] Classified intent: {state['intent']} with confidence {state['confidence']}")
    
    return state # Placeholder confidence

In [104]:
# Testing the intent classification with a sample state
state_tmp = AgentState({
    "customer_id": "12345",
    "query": "I'd like to get a credit limit increase.",
    "conversation_history": [],
    "intent": None,
    "confidence": 0.0,
    "response": "",
    "escalation_requested": False,
    "agent_response": ""
}
)

state_tmp = classify_intent(state_tmp)

[Intent classifier] Classified intent: account with confidence 1.0


In [105]:

def intent_classifier(state: AgentState) -> AgentState:
    """
    Ask for clarification when intent is ambiguous.
    For now, just return a message asking for clarification.
    """
    clarification_msg = "I'm not sure what you're asking. Could you clarify? Are you asking about your account, transactions, or general banking information?"
    state["response"] = clarification_msg
    print(f"[Clarification] Asking for more info")
    return state

In [106]:
def accounts_agent_node(state: AgentState) -> AgentState:
    """
    Handle account-related queries.
    Uses tools (get_account_balance) and RAG for policy questions.
    """
    # TODO: Implement actual agent with tools and RAG
    query = state["query"]
    customer_id = state["customer_id"]

    # Placeholder response
    state["agent_response"] = f"Checking account information for customer {customer_id}..."
    state["escalation_requested"] = False

    print(f"[Accounts Agent] Processing: {query}")
    return state

In [107]:

def transactions_agent_node(state: AgentState) -> AgentState:
    """
    Handle transaction-related queries.
    Uses get_transaction_history tool.
    """
    # TODO: Implement actual agent with transaction tools
    query = state["query"]
    customer_id = state["customer_id"]

    # Placeholder response
    state["agent_response"] = f"Retrieving transaction history for customer {customer_id}..."
    state["escalation_requested"] = False

    print(f"[Transactions Agent] Processing: {query}")
    return state

In [108]:
def general_enquiry_node(state: AgentState) -> AgentState:
    """
    Handle general banking questions and FAQs.
    Detects escalation requests (user wants to talk to a human).
    """
    # TODO: Implement LLM-based general enquiry handler
    query = state["query"].lower()

    # Simple escalation detection (TODO: replace with LLM-based detection)
    escalation_keywords = ["speak to agent", "human", "support", "escalate", "representative"]
    state["escalation_requested"] = any(kw in query for kw in escalation_keywords)

    if state["escalation_requested"]:
        state["escalation_reason"] = "User requested to speak with a human"

    state["agent_response"] = f"Handling general inquiry..."

    print(f"[General Enquiry] Processing: {query}, Escalation: {state['escalation_requested']}")
    return state

In [109]:
def escalation_check_node(state: AgentState) -> AgentState:
    """
    Check if escalation to human was requested during agent processing.
    Prepares the response based on escalation status.
    """
    if state["escalation_requested"]:
        state["response"] = f"[ESCALATING TO HUMAN] Reason: {state['escalation_reason']}\nAgent message: {state['agent_response']}"
        print(f"[Escalation Check] Escalation requested - routing to human")
    else:
        state["response"] = state["agent_response"]
        print(f"[Escalation Check] No escalation - using agent response")

    return state

In [110]:
def save_memory_node(state: AgentState) -> AgentState:
    """
    Save the turn to conversation history for memory/context in future turns.
    """
    # TODO: Implement actual memory persistence
    user_msg = HumanMessage(content=state["query"])
    ai_msg = AIMessage(content=state["response"])

    state["conversation_history"].append(user_msg)
    state["conversation_history"].append(ai_msg)

    print(f"[Memory] Saved turn to history (total turns: {len(state['conversation_history']) // 2})")
    return state

In [111]:
def route_by_confidence(state: AgentState) -> Literal["clarification", "route_to_agent"]:
    """
    Route based on intent confidence.
    If low confidence, ask for clarification. Otherwise, route to appropriate agent.
    """
    if state["confidence"] < 0.7:
        return "clarification"
    else:
        return "route_to_agent"

In [112]:
def route_to_agent(state: AgentState) -> Literal["accounts_agent", "transactions_agent", "general_enquiry"]:
    """
    Route to the appropriate agent based on intent.
    """
    if state["intent"] == "account":
        return "accounts_agent"
    elif state["intent"] == "transaction":
        return "transactions_agent"
    else:
        return "general_enquiry"


def route_after_clarification(state: AgentState) -> Literal["intent_classifier"]:
    """
    After clarification, loop back to intent classifier.
    In a real system, this would be triggered by user input.
    """
    return "intent_classifier"

In [113]:
def build_graph():
    """
    Construct and compile the LangGraph banking agent graph.
    """
    # Create the graph
    graph = StateGraph(AgentState)

    # Add all nodes
    graph.add_node("load_memory", load_memory_node)
    graph.add_node("intent_classifier", intent_classifier)
    graph.add_node("clarification", clarification_node)
    graph.add_node("accounts_agent", accounts_agent_node)
    graph.add_node("transactions_agent", transactions_agent_node)
    graph.add_node("general_enquiry", general_enquiry_node)
    graph.add_node("escalation_check", escalation_check_node)
    graph.add_node("save_memory", save_memory_node)

    # Add edges from start
    graph.add_edge(START, "load_memory")

    # Unconditional edges
    graph.add_edge("load_memory", "intent_classifier")
    graph.add_edge("escalation_check", "save_memory")
    graph.add_edge("save_memory", END)

    # Conditional edge from intent classifier (confidence check)
    graph.add_conditional_edges(
        "intent_classifier",
        route_by_confidence,
        {
            "clarification": "clarification",
            "route_to_agent": "route_to_agent_decision"
        }
    )

    # Conditional edge from clarification back to classifier
    # (In a real system, this would be triggered by user input)
    graph.add_edge("clarification", "intent_classifier")

    # Conditional edge to route to appropriate agent
    # Add a dummy node for routing decision
    graph.add_node("route_to_agent_decision", lambda state: state)
    graph.add_conditional_edges(
        "route_to_agent_decision",
        route_to_agent,
        {
            "accounts_agent": "accounts_agent",
            "transactions_agent": "transactions_agent",
            "general_enquiry": "general_enquiry"
        }
    )

    # All agents converge to escalation check
    graph.add_edge("accounts_agent", "escalation_check")
    graph.add_edge("transactions_agent", "escalation_check")
    graph.add_edge("general_enquiry", "escalation_check")

    # Compile the graph with memory
    checkpointer = InMemorySaver()
    app = graph.compile(checkpointer=checkpointer)

    return app

In [114]:
def run_agent(query: str, customer_id: str = "12345"):
    """
    Run the banking agent with a user query.
    """
    app = build_graph()

    initial_state = AgentState(
        customer_id=customer_id,
        query=query,
        conversation_history=[],
        intent="unknown",
        confidence=0.0,
        response="",
        escalation_requested=False,
        escalation_reason="",
        agent_response=""
    )

    # Use thread_id for conversation memory across multiple invocations
    config = {"configurable": {"thread_id": f"customer-{customer_id}"}}

    result = app.invoke(initial_state, config=config)

    return result

In [115]:
print("="*60)
print("Banking Agent System - Test Run")
print("="*60)

# Test 1: Account query
print("\n[Test 1] Account Query:")
result = run_agent("What's my account balance?", customer_id="12345")
print(f"Response: {result['response']}\n")

# Test 2: Transaction query
print("[Test 2] Transaction Query:")
result = run_agent("Show me my recent transactions", customer_id="12345")
print(f"Response: {result['response']}\n")

# Test 3: Escalation request
print("[Test 3] Escalation Request:")
result = run_agent("I want to speak to an agent", customer_id="12345")
print(f"Response: {result['response']}\n")

Banking Agent System - Test Run

[Test 1] Account Query:
[Memory] Loaded context for customer 12345
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarification] Asking for more info
[Clarificati

GraphRecursionError: Recursion limit of 10007 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT